In [ ]:
# @title 1. One-Click Setup (Auto-Restart Kernel)
import os
from IPython.display import clear_output

# Flag to avoid infinite restart loop
if not os.path.exists("/kaggle/working/setup_done.txt"):
    print("Installing dependencies... this will take a few minutes.")
    
    # 1. System Dependencies
    !apt-get update -y && apt-get install -y ffmpeg libgl1-mesa-glx
    
    # 2. Clone Repository
    if not os.path.exists("Wan2GP"):
        !git clone https://github.com/deepbeepmeep/Wan2GP.git
    
    os.chdir("/kaggle/working/Wan2GP")
    
    # 3. Python Dependencies (Optimized for T4)
    !pip install torch==2.6.0+cu126 torchvision==0.21.0+cu126 torchaudio==2.6.0+cu126 --index-url https://download.pytorch.org/whl/cu126
    !pip install -U "triton<3.3"
    !pip install sageattention==1.0.6
    !pip install -r requirements.txt
    !pip install "numpy<2"
    
    # Mark as done and restart
    with open("/kaggle/working/setup_done.txt", "w") as f:
        f.write("done")
    
    print("\n" + "="*50)
    print("INSTALLATION COMPLETE! RESTARTING KERNEL...")
    print("="*50)
    
    import os
    os._exit(00) # This forces the Kaggle kernel to restart
else:
    print("Dependencies already installed. Ready to run!")

In [ ]:
# @title 2. Storage Setup & Run Wan2GP
import os, shutil

# Ensure we are in the correct directory
os.chdir("/kaggle/working/Wan2GP")

# 1. Setup Storage (Using /kaggle/tmp for 70GB models)
BASE = "/kaggle/working/Wan2GP"
TMP  = "/kaggle/tmp/Wan2GP"
os.makedirs(TMP, exist_ok=True)

def setup_storage(name):
    src = os.path.join(BASE, name)
    dst = os.path.join(TMP, name)
    os.makedirs(dst, exist_ok=True)
    
    if os.path.islink(src):
        return
    if os.path.exists(src):
        # If it's a real folder, move contents and symlink
        for item in os.listdir(src):
            s_item = os.path.join(src, item)
            d_item = os.path.join(dst, item)
            if not os.path.exists(d_item):
                shutil.move(s_item, d_item)
        shutil.rmtree(src)
    
    os.symlink(dst, src)
    print(f"Storage linked: {name} -> {dst}")

setup_storage("ckpts")
# setup_storage("loras") # Uncomment if you use many LoRAs

# 2. Patch wgp.py to resolve symlinks correctly
!sed -i 's/local_dir = targetRoot/local_dir = os.path.realpath(targetRoot)/g' wgp.py

# 3. Clean start for outputs
!rm -rf outputs
!mkdir -p outputs

# 4. Launch WebUI
PORT = 7861
print("Starting Wan2GP...")
!python wgp.py --share --profile 4 --attention sage --listen --server-port {PORT}